In [1]:
# Core libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, classification_report, confusion_matrix)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
DATA_PATH = '../data/'

full_grouped = pd.read_csv(DATA_PATH + 'full_grouped.csv')
clean_complete = pd.read_csv(DATA_PATH + 'covid_19_clean_complete.csv')
country_latest = pd.read_csv(DATA_PATH + 'country_wise_latest.csv')
day_wise = pd.read_csv(DATA_PATH + 'day_wise.csv')
worldometer = pd.read_csv(DATA_PATH + 'worldometer_data.csv')

print("Datasets loaded:")
print(f"  full_grouped:   {full_grouped.shape}")
print(f"  clean_complete: {clean_complete.shape}")
print(f"  country_latest: {country_latest.shape}")
print(f"  day_wise:       {day_wise.shape}")
print(f"  worldometer:    {worldometer.shape}")

Datasets loaded:
  full_grouped:   (35156, 10)
  clean_complete: (49068, 10)
  country_latest: (187, 15)
  day_wise:       (188, 12)
  worldometer:    (209, 16)


In [3]:
# The main time-series dataset we'll focus on
full_grouped.head()

,Date,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,WHO Region
0,2020-01-22,Afghanistan,0,0,0,0,0,0,0,Eastern Mediterranean
1,2020-01-22,Albania,0,0,0,0,0,0,0,Europe
2,2020-01-22,Algeria,0,0,0,0,0,0,0,Africa
3,2020-01-22,Andorra,0,0,0,0,0,0,0,Europe
4,2020-01-22,Angola,0,0,0,0,0,0,0,Africa


In [4]:
full_grouped.info()

<class 'pandas.DataFrame'>
RangeIndex: 35156 entries, 0 to 35155
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Date            35156 non-null  str  
 1   Country/Region  35156 non-null  str  
 2   Confirmed       35156 non-null  int64
 3   Deaths          35156 non-null  int64
 4   Recovered       35156 non-null  int64
 5   Active          35156 non-null  int64
 6   New cases       35156 non-null  int64
 7   New deaths      35156 non-null  int64
 8   New recovered   35156 non-null  int64
 9   WHO Region      35156 non-null  str  
dtypes: int64(7), str(3)
memory usage: 2.7 MB


In [5]:
day_wise.head()

,Date,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,No. of countries
0,2020-01-22,555,17,28,510,0,0,0,3.06,5.05,60.71,6
1,2020-01-23,654,18,30,606,99,1,2,2.75,4.59,60.00,8
2,2020-01-24,941,26,36,879,287,8,6,2.76,3.83,72.22,9
3,2020-01-25,1434,42,39,1353,493,16,3,2.93,2.72,107.69,11
4,2020-01-26,2118,56,52,2010,684,14,13,2.64,2.46,107.69,13


In [6]:
country_latest.head()

,Country/Region,Confirmed,Deaths,Recovered,Active,New cases,New deaths,New recovered,Deaths / 100 Cases,Recovered / 100 Cases,Deaths / 100 Recovered,Confirmed last week,1 week change,1 week % increase,WHO Region
0,Afghanistan,36263,1269,25198,9796,106,10,18,3.50,69.49,5.04,35526,737,2.07,Eastern Mediterranean
1,Albania,4880,144,2745,1991,117,6,63,2.95,56.25,5.25,4171,709,17.00,Europe
2,Algeria,27973,1163,18837,7973,616,8,749,4.16,67.34,6.17,23691,4282,18.07,Africa
3,Andorra,907,52,803,52,10,0,0,5.73,88.53,6.48,884,23,2.60,Europe
4,Angola,950,41,242,667,18,1,0,4.32,25.47,16.94,749,201,26.84,Africa


In [7]:
worldometer.head()

,Country/Region,Continent,Population,TotalCases,NewCases,TotalDeaths,NewDeaths,TotalRecovered,NewRecovered,ActiveCases,"Serious,Critical",Tot Cases/1M pop,Deaths/1M pop,TotalTests,Tests/1M pop,WHO Region
0,USA,North America,331198130.00,5032179,NaN,162804.00,NaN,2576668.00,NaN,2292707.00,18296.00,15194.00,492.00,63139605.00,190640.00,Americas
1,Brazil,South America,212710692.00,2917562,NaN,98644.00,NaN,2047660.00,NaN,771258.00,8318.00,13716.00,464.00,13206188.00,62085.00,Americas
2,India,Asia,1381344997.00,2025409,NaN,41638.00,NaN,1377384.00,NaN,606387.00,8944.00,1466.00,30.00,22149351.00,16035.00,South-EastAsia
3,Russia,Europe,145940924.00,871894,NaN,14606.00,NaN,676357.00,NaN,180931.00,2300.00,5974.00,100.00,29716907.00,203623.00,Europe
4,South Africa,Africa,59381566.00,538184,NaN,9604.00,NaN,387316.00,NaN,141264.00,539.00,9063.00,162.00,3149807.00,53044.00,Africa
